# Assignment 4: Phishing Detection

**Evan Petersen** and **Malcolm Zartman**

https://colab.research.google.com/drive/10ZtvQQT7SuW0wpaSvHtU5PT9DnkIl_Nt?usp=sharing


In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
!pip install -q "datasets==2.21.0" "transformers==4.44.0" nltk scikit-learn

In [ ]:
import warnings, re, string
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk import ngrams

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.cuda.amp import autocast, GradScaler

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from transformers import BertTokenizer, BertModel

for r in ['punkt', 'punkt_tab', 'stopwords']:
    nltk.download(r, quiet=True)


In [ ]:
dataset = load_dataset('ealvaradob/phishing-dataset', 'urls', trust_remote_code=True)
df = pd.DataFrame(dataset['train'])
df.head()

In [ ]:
df_sample = df.sample(n=100_000, random_state=42)

X = df_sample['text'].astype(str).values
y = df_sample['label'].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)


In [ ]:
stemmer    = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text   = text.lower()
    text   = re.sub(f'[{re.escape(string.punctuation)}]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [stemmer.stem(t) for t in tokens if t not in stop_words and t.strip()]
    bi     = ['_'.join(g) for g in ngrams(tokens, 2)]
    return ' '.join(tokens + bi)

X_train_proc = list(map(preprocess, X_train))
X_val_proc   = list(map(preprocess, X_val))


In [ ]:
MAX_WORDS  = 10000
MAX_LEN    = 100
EMBED_DIM  = 64
N_FILTERS  = 64
UNITS      = 64
DROPOUT    = 0.3
EPOCHS     = 5
BATCH_SIZE = 128
PADDING    = 'post'
TRUNCATING = 'post'
LR         = 1e-3

tok = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tok.fit_on_texts(X_train_proc)

X_tr_np = pad_sequences(tok.texts_to_sequences(X_train_proc),
                        maxlen=MAX_LEN, padding=PADDING, truncating=TRUNCATING)
X_vl_np = pad_sequences(tok.texts_to_sequences(X_val_proc),
                        maxlen=MAX_LEN, padding=PADDING, truncating=TRUNCATING)

X_tr = torch.tensor(X_tr_np, dtype=torch.long)
X_vl = torch.tensor(X_vl_np, dtype=torch.long)
y_tr = torch.tensor(y_train, dtype=torch.float32)
y_vl = torch.tensor(y_val,   dtype=torch.float32)


In [ ]:
aucs   = {}
scaler = GradScaler()

def plot_roc(name, y_true, y_pred):
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, label=f'AUC = {auc:.4f}')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('FPR'); plt.ylabel('TPR')
    plt.title(f'ROC - {name}')
    plt.legend(); plt.tight_layout(); plt.show()
    return auc

def train_eval(model, X_t, y_t, X_v, y_v, epochs=EPOCHS, bs=BATCH_SIZE, lr=LR):
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=bs, shuffle=True)
    opt    = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()
    model.to(device)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            with autocast():
                pred = model(xb).squeeze(1)
            loss = loss_fn(pred.float(), yb)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
    model.eval()
    with torch.no_grad():
        with autocast():
            preds = model(X_v.to(device)).squeeze(1)
        preds = preds.float().cpu().numpy()
    return preds


In [ ]:
class DenseModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb  = nn.Embedding(MAX_WORDS, EMBED_DIM, padding_idx=0)
        self.drop = nn.Dropout(DROPOUT)
        self.fc1  = nn.Linear(EMBED_DIM * MAX_LEN, UNITS)
        self.fc2  = nn.Linear(UNITS, 1)
    def forward(self, x):
        x = self.emb(x).view(x.size(0), -1)
        return torch.sigmoid(self.fc2(self.drop(F.relu(self.fc1(x)))))

preds = train_eval(DenseModel(), X_tr, y_tr, X_vl, y_vl)
aucs['Dense'] = plot_roc('Dense', y_val, preds)

In [ ]:
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb  = nn.Embedding(MAX_WORDS, EMBED_DIM, padding_idx=0)
        self.conv = nn.Conv1d(EMBED_DIM, N_FILTERS, kernel_size=3, padding=1)
        self.drop = nn.Dropout(DROPOUT)
        self.fc   = nn.Linear(N_FILTERS, 1)
    def forward(self, x):
        x = self.emb(x).permute(0, 2, 1)
        x = F.relu(self.conv(x)).max(dim=2).values
        return torch.sigmoid(self.fc(self.drop(x)))

preds = train_eval(CNNModel(), X_tr, y_tr, X_vl, y_vl)
aucs['CNN'] = plot_roc('CNN', y_val, preds)


In [ ]:
class RNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb  = nn.Embedding(MAX_WORDS, EMBED_DIM, padding_idx=0)
        self.rnn  = nn.RNN(EMBED_DIM, UNITS, batch_first=True)
        self.drop = nn.Dropout(DROPOUT)
        self.fc   = nn.Linear(UNITS, 1)
    def forward(self, x):
        _, h = self.rnn(self.emb(x))
        return torch.sigmoid(self.fc(self.drop(h.squeeze(0))))

preds = train_eval(RNNModel(), X_tr, y_tr, X_vl, y_vl)
aucs['Simple RNN'] = plot_roc('Simple RNN', y_val, preds)


In [ ]:
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb  = nn.Embedding(MAX_WORDS, EMBED_DIM, padding_idx=0)
        self.lstm = nn.LSTM(EMBED_DIM, UNITS, batch_first=True)
        self.drop = nn.Dropout(DROPOUT)
        self.fc   = nn.Linear(UNITS, 1)
    def forward(self, x):
        _, (h, _) = self.lstm(self.emb(x))
        return torch.sigmoid(self.fc(self.drop(h.squeeze(0))))

preds = train_eval(LSTMModel(), X_tr, y_tr, X_vl, y_vl)
aucs['LSTM'] = plot_roc('LSTM', y_val, preds)

In [ ]:
class BiLSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb  = nn.Embedding(MAX_WORDS, EMBED_DIM, padding_idx=0)
        self.lstm = nn.LSTM(EMBED_DIM, UNITS, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(DROPOUT)
        self.fc   = nn.Linear(UNITS * 2, 1)
    def forward(self, x):
        _, (h, _) = self.lstm(self.emb(x))
        h = torch.cat([h[0], h[1]], dim=1)
        return torch.sigmoid(self.fc(self.drop(h)))

preds = train_eval(BiLSTMModel(), X_tr, y_tr, X_vl, y_vl)
aucs['Bi-LSTM'] = plot_roc('Bi-LSTM', y_val, preds)


In [ ]:
BERT_MAX_LEN = 64
BERT_N_TRAIN = 10_000
BERT_N_VAL   = 2_000
BERT_EPOCHS  = 3
BERT_BATCH   = 64

rng    = np.random.default_rng(42)
tr_idx = rng.choice(len(X_train), size=BERT_N_TRAIN, replace=False)
vl_idx = rng.choice(len(X_val),   size=BERT_N_VAL,   replace=False)

X_bert_train, y_bert_train = X_train[tr_idx], y_train[tr_idx]
X_bert_val,   y_bert_val   = X_val[vl_idx],   y_val[vl_idx]

bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_base      = BertModel.from_pretrained('bert-base-uncased').to(device)
bert_base.eval()
for p in bert_base.parameters():
    p.requires_grad = False

def extract_bert(texts, batch_size=64):
    cls_list, seq_list = [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = bert_tokenizer(list(batch), max_length=BERT_MAX_LEN,
                               truncation=True, padding='max_length', return_tensors='pt')
        enc   = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad(), autocast():
            out = bert_base(**enc)
        cls_list.append(out.last_hidden_state[:, 0, :].float().cpu())
        seq_list.append(out.last_hidden_state.float().cpu())
    return torch.cat(cls_list), torch.cat(seq_list)

cls_tr, seq_tr = extract_bert(X_bert_train)
cls_vl, seq_vl = extract_bert(X_bert_val)

y_btr = torch.tensor(y_bert_train, dtype=torch.float32)
y_bvl = torch.tensor(y_bert_val,   dtype=torch.float32)


def train_eval_bert(model, X_t, y_t, X_v, y_v,
                    epochs=BERT_EPOCHS, bs=BERT_BATCH, lr=1e-3):
    loader  = DataLoader(TensorDataset(X_t, y_t), batch_size=bs, shuffle=True)
    opt     = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()
    model.to(device)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            with autocast():
                pred = model(xb).squeeze(1)
            loss = loss_fn(pred.float(), yb)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
    model.eval()
    with torch.no_grad():
        with autocast():
            preds = model(X_v.to(device)).squeeze(1)
        preds = preds.float().cpu().numpy()
    return preds

In [ ]:
class BertDenseHead(nn.Module):
    def __init__(self, units=UNITS, dropout=DROPOUT):
        super().__init__()
        self.fc1  = nn.Linear(768, units)
        self.fc2  = nn.Linear(units, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return torch.sigmoid(self.fc2(self.drop(F.relu(self.fc1(x)))))

preds = train_eval_bert(BertDenseHead(), cls_tr, y_btr, cls_vl, y_bvl)
aucs['BERT+Dense'] = plot_roc('BERT+Dense', y_bert_val, preds)

In [ ]:
class BertCNNHead(nn.Module):
    def __init__(self, n_filters=N_FILTERS, dropout=DROPOUT):
        super().__init__()
        self.conv = nn.Conv1d(768, n_filters, kernel_size=3, padding=1)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(n_filters, 1)
    def forward(self, x):
        x = F.relu(self.conv(x.permute(0, 2, 1))).max(dim=2).values
        return torch.sigmoid(self.fc(self.drop(x)))

preds = train_eval_bert(BertCNNHead(), seq_tr, y_btr, seq_vl, y_bvl)
aucs['BERT+CNN'] = plot_roc('BERT+CNN', y_bert_val, preds)


In [ ]:
class BertRNNHead(nn.Module):
    def __init__(self, units=UNITS, dropout=DROPOUT):
        super().__init__()
        self.rnn  = nn.RNN(768, units, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(units, 1)
    def forward(self, x):
        _, h = self.rnn(x)
        return torch.sigmoid(self.fc(self.drop(h.squeeze(0))))

preds = train_eval_bert(BertRNNHead(), seq_tr, y_btr, seq_vl, y_bvl)
aucs['BERT+Simple RNN'] = plot_roc('BERT+Simple RNN', y_bert_val, preds)

In [ ]:
class BertLSTMHead(nn.Module):
    def __init__(self, units=UNITS, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(768, units, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(units, 1)
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return torch.sigmoid(self.fc(self.drop(h.squeeze(0))))

preds = train_eval_bert(BertLSTMHead(), seq_tr, y_btr, seq_vl, y_bvl)
aucs['BERT+LSTM'] = plot_roc('BERT+LSTM', y_bert_val, preds)

In [ ]:
class BertBiLSTMHead(nn.Module):
    def __init__(self, units=UNITS, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(768, units, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(units * 2, 1)
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        h = torch.cat([h[0], h[1]], dim=1)
        return torch.sigmoid(self.fc(self.drop(h)))

preds = train_eval_bert(BertBiLSTMHead(), seq_tr, y_btr, seq_vl, y_bvl)
aucs['BERT+Bi-LSTM'] = plot_roc('BERT+Bi-LSTM', y_bert_val, preds)

In [ ]:
tuned_aucs = {}

E, U, NF, D, LR2 = 128, 128, 128, 0.2, 5e-4

class DenseT(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb=nn.Embedding(MAX_WORDS,E,padding_idx=0); self.drop=nn.Dropout(D)
        self.fc1=nn.Linear(E*MAX_LEN,U); self.fc2=nn.Linear(U,1)
    def forward(self,x):
        return torch.sigmoid(self.fc2(self.drop(torch.relu(self.fc1(self.emb(x).view(x.size(0),-1))))))

class CNNT(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb=nn.Embedding(MAX_WORDS,E,padding_idx=0); self.conv=nn.Conv1d(E,NF,3,padding=1)
        self.drop=nn.Dropout(D); self.fc=nn.Linear(NF,1)
    def forward(self,x):
        return torch.sigmoid(self.fc(self.drop(torch.relu(self.conv(self.emb(x).permute(0,2,1))).max(2).values)))

class RNNT(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb=nn.Embedding(MAX_WORDS,E,padding_idx=0); self.rnn=nn.RNN(E,U,batch_first=True)
        self.drop=nn.Dropout(D); self.fc=nn.Linear(U,1)
    def forward(self,x):
        _,h=self.rnn(self.emb(x)); return torch.sigmoid(self.fc(self.drop(h.squeeze(0))))

class LSTMT(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb=nn.Embedding(MAX_WORDS,E,padding_idx=0); self.lstm=nn.LSTM(E,U,batch_first=True)
        self.drop=nn.Dropout(D); self.fc=nn.Linear(U,1)
    def forward(self,x):
        _,(h,_)=self.lstm(self.emb(x)); return torch.sigmoid(self.fc(self.drop(h.squeeze(0))))

class BiLSTMT(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb=nn.Embedding(MAX_WORDS,E,padding_idx=0); self.lstm=nn.LSTM(E,U,batch_first=True,bidirectional=True)
        self.drop=nn.Dropout(D); self.fc=nn.Linear(U*2,1)
    def forward(self,x):
        _,(h,_)=self.lstm(self.emb(x)); h=torch.cat([h[0],h[1]],1)
        return torch.sigmoid(self.fc(self.drop(h)))

nlp_tuned = [('Dense',DenseT()),('CNN',CNNT()),('Simple RNN',RNNT()),('LSTM',LSTMT()),('Bi-LSTM',BiLSTMT())]

for name, model in nlp_tuned:
    preds = train_eval(model, X_tr, y_tr, X_vl, y_vl, epochs=8, bs=256, lr=LR2)
    tuned_aucs[name] = plot_roc(f'Tuned {name}', y_val, preds)


In [ ]:
U2, F2, D2, LR3 = 256, 128, 0.2, 3e-4

class BertDenseT(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1=nn.Linear(768,U2); self.fc2=nn.Linear(U2,1); self.drop=nn.Dropout(D2)
    def forward(self,x): return torch.sigmoid(self.fc2(self.drop(F.relu(self.fc1(x)))))

class BertCNNT(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv=nn.Conv1d(768,F2,3,padding=1); self.drop=nn.Dropout(D2); self.fc=nn.Linear(F2,1)
    def forward(self,x): return torch.sigmoid(self.fc(self.drop(F.relu(self.conv(x.permute(0,2,1))).max(2).values)))

class BertRNNT(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn=nn.RNN(768,U2,batch_first=True); self.drop=nn.Dropout(D2); self.fc=nn.Linear(U2,1)
    def forward(self,x):
        _,h=self.rnn(x); return torch.sigmoid(self.fc(self.drop(h.squeeze(0))))

class BertLSTMT(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm=nn.LSTM(768,U2,batch_first=True); self.drop=nn.Dropout(D2); self.fc=nn.Linear(U2,1)
    def forward(self,x):
        _,(h,_)=self.lstm(x); return torch.sigmoid(self.fc(self.drop(h.squeeze(0))))

class BertBiLSTMT(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm=nn.LSTM(768,U2,batch_first=True,bidirectional=True); self.drop=nn.Dropout(D2); self.fc=nn.Linear(U2*2,1)
    def forward(self,x):
        _,(h,_)=self.lstm(x); h=torch.cat([h[0],h[1]],1); return torch.sigmoid(self.fc(self.drop(h)))

bert_tuned = [
    ('BERT+Dense',      BertDenseT(),   cls_tr, cls_vl),
    ('BERT+CNN',        BertCNNT(),     seq_tr, seq_vl),
    ('BERT+Simple RNN', BertRNNT(),     seq_tr, seq_vl),
    ('BERT+LSTM',       BertLSTMT(),    seq_tr, seq_vl),
    ('BERT+Bi-LSTM',    BertBiLSTMT(),  seq_tr, seq_vl),
]

for name, model, Xt, Xv in bert_tuned:
    preds = train_eval_bert(model, Xt, y_btr, Xv, y_bvl, epochs=5, bs=128, lr=LR3)
    tuned_aucs[name] = plot_roc(f'Tuned {name}', y_bert_val, preds)


In [ ]:
all_names  = list(aucs.keys())
base_vals  = [aucs[n] for n in all_names]
tuned_vals = [tuned_aucs.get(n, float('nan')) for n in all_names]

x_pos = range(len(all_names))
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar([p - 0.2 for p in x_pos], base_vals,  0.4, label='Baseline')
ax.bar([p + 0.2 for p in x_pos], tuned_vals, 0.4, label='Tuned')
ax.set_xticks(list(x_pos))
ax.set_xticklabels(all_names, rotation=30, ha='right')
ax.set_ylabel('ROC-AUC')
ax.set_title('Baseline vs Tuned ROC-AUC - All 10 Models')
ax.legend(); plt.tight_layout(); plt.show()
